In [1]:
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import nltk
from pathlib import Path
from utils import dataset_path

In [2]:
data_path = Path("data")
count = 0
for mail_path in os.listdir(dataset_path):
    for path in os.listdir(dataset_path/ Path(mail_path) / data_path):
        if path.endswith('.csv'):
            if count == 0:
                df = pd.read_csv(dataset_path/ Path(mail_path) / data_path / Path(path))
                count += 1
            else:
                df = pd.concat([df, pd.read_csv(dataset_path/ Path(mail_path) / data_path / Path(path))], ignore_index=True)
print(df.value_counts('Reachable'))
print("Text is None: ",len(df[df['Text'].isna()]))
df = df.dropna(subset=['Text'])
df = df[df['Reachable']]
df = df.reset_index(drop=True)
print("Total number of texts to process: ",len(df))

Reachable
True     2039
False     647
Name: count, dtype: int64
Text is None:  748
Total number of texts to process:  1938


In [3]:
df.drop(['Links', 'Reachable','Code'], axis=1, inplace=True)

from transformers import BertTokenizer

# Load the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Text to tokenize
text = "Sample text to tokenize."

# Tokenize the text
tokens = tokenizer.tokenize(text)
print("Tokens:", tokens)

# Count the number of tokens
num_tokens = len(tokens)

print("Number of tokens:", num_tokens)

In [4]:
df = df.iloc[:200]

In [3]:
df

,Links,Reachable,Code,Text,Date
0,https://www.routledge.com/Routledge-Internatio...,True,200,1st Edition\nRoutledge International Handbook ...,"Wed, 18 May 2016 09:44:26 -0000"
1,http://www.gala.fr/l_actu/culture/la_fille_inc...,True,200,"Pour la septième fois, Luc et Jean-Pierre Dard...","Fri, 20 May 2016 00:13:52 -0000"
2,http://ifp.nyu.edu/2016/journal-article-abstra...,True,200,Unraveling the Slut Narrative: Gender Constrai...,"Thu, 26 May 2016 09:44:26 -0000"
3,http://www.actuabd.com/Platinum-End-T1-Par-Tsu...,True,200,C’est le troisième gros lancement manga de l’a...,"Sat, 28 May 2016 08:14:14 -0000"
4,http://next.liberation.fr/cinema/2016/05/31/a-...,True,200,A War explore en deux temps les opérations mil...,"Wed, 01 Jun 2016 00:14:17 -0000"
...,...,...,...,...,...
1933,https://voi.id/en/news/377517,True,200,JAKARTA - The Corruption Eradication Commissio...,"Wed, 01 May 2024 12:40:50 -0000"
1934,https://dailyguardian.com.ph/first-light-a-shi...,True,200,"By Christine Marie Lim Magpile, LPT\nEast Meet...","Wed, 01 May 2024 12:40:50 -0000"
1935,https://www.thisislocallondon.co.uk/news/24290...,True,200,‘The Hunger Games’ book review\nSuzanne Collin...,"Wed, 01 May 2024 12:40:50 -0000"
1936,https://www.celluloidz.com/psycho/decouverte-s...,True,200,Au gré des swipes sur votre application de ren...,"Wed, 01 May 2024 15:13:52 -0000"


In [4]:
df.to_csv('google_alerts/processed_data.csv', index=False)

In [6]:
stop_words = nltk.corpus.stopwords.words('french') + nltk.corpus.stopwords.words('english')
count_vect = CountVectorizer(max_df=0.5, min_df=0.1, stop_words=stop_words)
doc_term_matrix = count_vect.fit_transform(df['Text'].values)

In [7]:
nb_topics = 22


In [8]:
LDA = LatentDirichletAllocation(n_components=nb_topics, random_state=42)
LDA.fit(doc_term_matrix)

LatentDirichletAllocation(n_components=22, random_state=42)

In [9]:
first_topic = LDA.components_[0]
top_topic_words = first_topic.argsort()[-10:]
for i in top_topic_words:
    print(count_vect.get_feature_names_out()[i])

fin
militaire
depuis
très
pays
peut
ans
sans
contre
guerre


In [10]:
for i,topic in enumerate(LDA.components_):
    print(f'Top 10 words for topic #{i}:')
    print([count_vect.get_feature_names_out()[i] for i in topic.argsort()[-15:]])
    print('\n')

Top 10 words for topic #0:
['jour', 'jusqu', 'vie', 'attaque', 'avoir', 'fin', 'militaire', 'depuis', 'très', 'pays', 'peut', 'ans', 'sans', 'contre', 'guerre']


Top 10 words for topic #1:
['cela', 'choix', 'loi', 'selon', 'exemple', 'manière', 'peut', 'non', 'où', 'enfant', 'personnes', 'sans', 'décision', 'enfants', 'vie']


Top 10 words for topic #2:
['forme', 'nouvelle', 'décisions', 'sortie', 'personnages', 'septembre', 'histoire', 'monde', 'réalisé', 'états', 'écrit', 'unis', 'jeu', '2023', 'vidéo']


Top 10 words for topic #3:
['nouvelle', 'autres', 'faut', 'mis', 'premier', 'encore', 'effet', 'puis', 'rien', 'information', 'maintenant', 'jour', 'dit', '2023', 'vie']


Top 10 words for topic #4:
['selon', 'personnes', 'trouver', 'cela', 'groupe', 'peut', 'vie', 'sauver', 'nouvelle', 'pouvez', 'jeu', 'leurs', 'deux', 'également', 'choix']


Top 10 words for topic #5:
['celle', 'avoir', 'jusqu', 'pendant', 'millions', 'ans', 'prendre', 'fils', 'personne', 'ville', 'vie', 'mère', 

# OTHER METHOD

In [11]:
nb_topics = 25

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vect = TfidfVectorizer(max_df=0.8, min_df=2, stop_words=stop_words)
doc_term_matrix = tfidf_vect.fit_transform(df['Text'].values.astype('U'))

In [13]:
from sklearn.decomposition import NMF

nmf = NMF(n_components=nb_topics, random_state=42)
nmf.fit(doc_term_matrix )

C:\Users\Emile\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\decomposition\_nmf.py:1770: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


NMF(n_components=25, random_state=42)

In [14]:
components_df = pd.DataFrame(nmf.components_, columns=tfidf_vect.get_feature_names_out())
components_df

,00,000,01,02,03,04,05,06,07,08,...,événements,être,êtres,île,îles,œil,œuf,œufs,œuvre,œuvres
0,0.005385,0.170873,0.000000,0.000000,0.002367,0.000000,0.000000,0.000000,0.009052,0.030945,...,0.000000,0.343425,0.026917,0.019762,0.000564,0.002094,0.019215,0.001442,0.000000,0.010220
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000045,0.000035,...,0.000007,0.000031,0.000021,0.000000,0.000000,0.000000,0.000031,0.000078,0.000000,0.000000
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.002402,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.000000,0.113389,0.000000,0.000000,0.003702,0.000000,0.000000,0.000000,0.010781,0.000000,...,0.000000,0.057061,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.016293,0.067119,0.000000,0.004248,0.000000,0.008282,0.000000,0.000000,0.001193,0.000000
5,0.012430,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.060986,0.025861,0.014757,0.000000,0.034287,0.000000,0.000000,0.000000,0.008632,0.000000
6,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000847,0.000000,0.000000,0.018103,0.000000,0.000000,0.000000,0.000000,0.000000
7,0.000304,0.000000,0.000000,0.000000,0.000000,0.000000,0.000402,0.000941,0.000000,0.000000,...,0.000000,0.000667,0.000121,0.000000,0.000088,0.000000,0.000000,0.000269,0.000000,0.000000
8,0.032368,0.000000,0.070916,0.018125,0.404410,1.088310,0.174061,0.085087,0.085993,0.081749,...,0.000000,0.005413,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.004016,0.000000
9,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.020797,0.000000,0.000000,0.000000,...,0.084771,0.052788,0.001770,0.000000,0.000000,0.027914,0.043985,0.000000,0.042358,0.019588


In [15]:
for topic in range(components_df.shape[0]):
    tmp = components_df.iloc[topic]
    print(f'For topic {topic+1} the words with the highest value are:')
    print(tmp.nlargest(10))
    print('\n')

For topic 1 the words with the highest value are:
plus     0.572026
si       0.520067
faire    0.422179
tout     0.400702
ça       0.376310
être     0.343425
cela     0.334194
comme    0.324634
fait     0.297794
dit      0.295537
Name: 0, dtype: float64


For topic 2 the words with the highest value are:
access           0.560786
apologize        0.560786
available        0.560786
inconvenience    0.560786
longer           0.560786
removed          0.560786
trying           0.560786
content          0.528514
ellie            0.000493
floride          0.000477
Name: 1, dtype: float64


For topic 3 the words with the highest value are:
hoult         0.650839
jouera        0.637429
juré          0.588060
eastwood      0.456718
collette      0.348354
sutherland    0.342023
procès        0.339205
deutch        0.305158
jury          0.278686
clint         0.246055
Name: 2, dtype: float64


For topic 4 the words with the highest value are:
hamas          0.827706
israël         0.627708
gaza

In [16]:
theme_list =  ['rien', 'rien', 'news/journal', 'bug','2024', 'site_down', 'femme_avortement_société', 'manga_japon',  'Bollywood', 'ethique', 'guerre_russie', 'soap_opera', 'petrole_climat_geothermie', 'justice', 'russie_anglais', 'série', 'rien', 'IA_voitures_autonomes', 'israel_gaza', 'films' , 'coree', 'music', 'energie', 'police_medicale', 'cdm_qatar', 'oppenheimer_manhattan', 'medical', 'euthanasie_souffrance', 'covid' , 'jeux_video_fantaisie' ]

In [17]:
features = nmf.transform(doc_term_matrix)

In [18]:
len(features)

200

In [19]:
len(features[0])

25

In [20]:
print(np.argmax(features[0]))

20


In [21]:
sorted_topics = [np.argsort(features[i]) for i in range(len(features))]
print(sorted_topics)


[array([10,  1,  2,  4, 17,  7, 15, 22, 11, 23,  3,  6,  8, 12,  5, 19, 21,
       18, 14,  9, 13, 16,  0, 24, 20], dtype=int64), array([12, 15, 14, 19, 10,  9, 16,  8,  6, 20, 21,  3,  1,  7, 18,  2, 11,
       13,  0, 24,  5, 23, 22, 17,  4], dtype=int64), array([ 0, 22, 21, 20, 19, 18, 17, 16, 15, 14, 13, 23, 12, 10,  9,  8,  6,
        5,  4,  3,  2,  1, 11, 24,  7], dtype=int64), array([24, 16, 21, 14, 13, 11, 10, 18,  9,  7, 22,  5,  2,  1,  8, 19,  3,
       20, 23,  4, 12,  6, 17, 15,  0], dtype=int64), array([ 0, 22, 21, 20, 19, 18, 17, 16, 15, 14, 13, 23, 12, 10,  9,  8,  6,
        5,  4,  3,  2,  1, 11, 24,  7], dtype=int64), array([ 0, 22, 20, 18, 17, 13, 23, 24,  1,  5,  3,  4, 10,  8, 11,  6,  2,
       12, 19, 15, 14, 16, 21,  9,  7], dtype=int64), array([ 0, 22, 21, 20, 19, 18, 17, 16, 15, 14, 13, 23, 24, 10,  9,  8,  7,
        6,  5,  4,  3,  2,  1, 11, 12], dtype=int64), array([12, 22, 21, 20, 19, 18, 17, 15, 14, 13, 23, 24, 10,  9,  8,  7,  6,
        4,  3,  2,  1

In [122]:
first_topic = []
second_topic = []
third_topic = []
for tops in sorted_topics:
    first_topic.append(theme_list[tops[-1]])
    second_topic.append(theme_list[tops[-2]])
    third_topic.append(theme_list[tops[-3]])

df['First_topic'] = first_topic
df['Second_topic'] = second_topic
df['Third_topic'] = third_topic

In [123]:
df

,Links,Reachable,Code,Text,First_topic,Second_topic,Third_topic
0,https://www.programme-television.org/news-tv/I...,True,200,Télé\nIci tout commence : Mehdi sous le choc f...,rien,série,films
1,https://www.philomag.com/articles/leurope-et-l...,True,200,L’Europe et la guerre en Ukraine vues par un p...,guerre_russie,rien,oppenheimer_manhattan
2,https://nouvelles-dujour.com/jerrod-carmichael...,True,200,Account Suspended\nThis Account has been suspe...,site_down,jeux_video_fantaisie,covid
3,https://lbe.news/les-gagnants-des-80e-golden-g...,True,200,Unknown HostThis server does not serve the hos...,bug,vie_famille,femme_avortement_société
4,https://www.latribune.ca/2023/01/13/captivant-...,True,200,Deux couples aux antipodes des privilèges fina...,films,rien,news/journal
...,...,...,...,...,...,...,...
679,http://www.critictoo.com/bilans-de-saisons/per...,True,200,Tout commençait par un crash. La famille Robin...,série,rien,IA_voitures_autonomes
680,https://asialyst.com/fr/2020/01/17/inde-bangla...,True,200,Films d'Asie du Sud\nUne semaine de cinéma ind...,films,covid,guerre_russie
681,https://www.lexpress.fr/actualite/societe/reto...,True,200,"Ce retour aux 90 kilomètres-heure, ils l'ont p...",rien,horloge_enigme,covid
682,https://www.lemonde.fr/culture/article/2020/01...,True,200,ARTE - MARDI 21 JANVIER À 20 H 50 - DOCUMENTAI...,israel_gaza,covid,rien


In [126]:
df = df[df['First_topic'] != 'bug'][df['First_topic'] != 'site_down']

C:\Users\Emile\AppData\Local\Temp\ipykernel_37708\549096651.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[df['First_topic'] != 'bug'][df['First_topic'] != 'site_down']


In [127]:
df.to_excel("results/Topics.xlsx", index=False)

In [128]:
len(df)

660